In [ ]:
import json
import os
from pathlib import Path

import pandas as pd
from IPython.display import display

In [ ]:
TASK = Path.cwd().parent
CONFIG = json.loads((TASK.parent/'config.json').read_text())
INPUT, OUTPUT = TASK/'input', TASK/'output'
RAW = Path(os.environ.get('AARC_RAW_DIR', Path.home()/'Desktop/SHL/SHL_aarc')).expanduser().resolve()
INSTITUTIONS = Path(os.environ.get('AARC_INSTITUTIONS', RAW/'phd_granting_institutions.csv')).expanduser().resolve()
ASSETS = ['department-faculty', 'program-faculty', 'medical-faculty',
          'professorial_rank', 'articles', 'conference-proceedings', 'taxonomy']

In [ ]:
sources = {name+'.csv': RAW/(name+'.csv') for name in ASSETS}
sources['phd_granting_institutions.csv'] = INSTITUTIONS
missing = [str(path) for path in sources.values() if not path.is_file()]
if missing:
    raise FileNotFoundError('Missing AARC inputs:\n' + '\n'.join(missing))

In [ ]:
institutions = pd.read_csv(INSTITUTIONS, usecols=['InstitutionId']).InstitutionId
if institutions.isna().any() or len(institutions) != CONFIG['institution_count'] or not institutions.is_unique:
    raise ValueError(f"Expected the original {CONFIG['institution_count']} unique employing institution IDs")

taxonomy = pd.read_csv(sources['taxonomy.csv'], usecols=['TaxonomyLevel01Id', 'TaxonomyLevel01Name'])
taxon_ids = {code for field in CONFIG['fields'] for code in field['aarc_ids']}
missing_ids = taxon_ids - set(taxonomy.TaxonomyLevel01Id.dropna())
if missing_ids:
    raise ValueError(f'Official AARC taxonomy is missing configured IDs: {sorted(missing_ids)}')

In [ ]:
# link restricted assets without copying them
for name, source in sources.items():
    for directory, target in [(INPUT, source), (OUTPUT, Path('../input')/name)]:
        link = directory/name
        if link.is_symlink() and link.readlink() != target:
            link.unlink()
        if not link.is_symlink():
            link.symlink_to(target)
display(pd.DataFrame({'asset': list(sources)}))